# Week 94, Day 3: The CPython developer guide

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w094d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import json

print(json.__file__.replace("\\", "/").split("/")[-2:])
print(json.dumps.__module__, json.JSONDecoder.__module__)

> **Key idea.** CPython's map: `Lib/` (Python stdlib), `Lib/test/` (tests), `Doc/` (reST docs), `Objects/` (built-in types), `Python/` (interpreter), `Modules/` (C modules), `Misc/NEWS.d/` (NEWS). Workflow: issue, fork and branch, change with a test, NEWS via blurb (or skip news), PR titled `gh-N: ...`, CLA, review and CI, merge, backports. Features go to `main`; bug fixes are backported to bugfix branches; security fixes to security branches too. Check the current devguide.

### Exercise: Where does it live?

Write `locate(kind, name)`: `"library"` gives `Lib/NAME.py`; `"package"` gives `Lib/NAME/`; `"test"` gives `Lib/test/test_NAME.py`; `"docs"` gives `Doc/library/NAME.rst`; `"type"` gives `Objects/NAMEobject.c`; `"c module"` gives `Modules/NAME.c`; `"builtin"` gives `Python/bltinmodule.c` whatever the name. Raise `ValueError("unknown kind: KIND")` otherwise.

In [ ]:
def locate(kind, name):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_locate():\n    \"CPython's map\"\n    assert locate(\"library\", \"textwrap\") == \"Lib/textwrap.py\" and locate(\"package\", \"json\") == \"Lib/json/\"\n    assert locate(\"test\", \"textwrap\") == \"Lib/test/test_textwrap.py\" and locate(\"docs\", \"json\") == \"Doc/library/json.rst\"\n    assert locate(\"type\", \"list\") == \"Objects/listobject.c\" and locate(\"c module\", \"_json\") == \"Modules/_json.c\"\n    assert locate(\"builtin\", \"len\") == \"Python/bltinmodule.c\"\n    try:\n        locate(\"wiki\", \"x\")\n    except ValueError as e:\n        assert str(e) == \"unknown kind: wiki\"\n    else:\n        raise AssertionError(\"wiki\")")

### Exercise: Where blurb puts it

Write `news_path(section, issue, when, nonce)`: `when` is a `datetime`. Return `Misc/NEWS.d/next/SECTION/YYYY-MM-DD-HH-MM-SS.gh-issue-ISSUE.NONCE.rst`. Raise `ValueError` with `"unknown section: S"` if the section isn't one of `Security`, `Core_and_Builtins`, `Library`, `Documentation`, `Tests`, `Build`, `Windows`, `macOS`, `IDLE`, `Tools-Demos` and `C_API`; `"issue must be a positive number"` unless `issue` is an int above 0; and `"nonce must be 6 letters or digits"`.

In [ ]:
from datetime import datetime

def news_path(section, issue, when, nonce):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from datetime import datetime\n\ndef test_path():\n    \"blurb's naming\"\n    when = datetime(2026, 11, 20, 14, 3, 12)\n    assert news_path(\"Library\", 140123, when, \"AbCdEf\") == \"Misc/NEWS.d/next/Library/2026-11-20-14-03-12.gh-issue-140123.AbCdEf.rst\"\n    assert news_path(\"Core_and_Builtins\", 7, when, \"x1Y2z3\").startswith(\"Misc/NEWS.d/next/Core_and_Builtins/2026-11-20-14-03-12.gh-issue-7.\")\n\ndef test_bad():\n    \"checked\"\n    when = datetime(2026, 11, 20)\n    for args, message in [((\"Libraries\", 1, when, \"AbCdEf\"), \"unknown section: Libraries\"), ((\"Library\", 0, when, \"AbCdEf\"), \"issue must be a positive number\"),\n                          ((\"Library\", \"12\", when, \"AbCdEf\"), \"issue must be a positive number\"), ((\"Library\", 1, when, \"Ab-dEf\"), \"nonce must be 6 letters or digits\"),\n                          ((\"Library\", 1, when, \"AbC\"), \"nonce must be 6 letters or digits\")]:\n        try:\n            news_path(*args)\n        except ValueError as e:\n            assert str(e) == message\n        else:\n            raise AssertionError(message)")

### Exercise: A NEWS entry for users

Write `news_problems(text)`: after stripping the text, report in this order: `"empty"` (and nothing else) if nothing's left; `"more than one paragraph"` if it contains a blank line; `"should start with a capital letter or a role"` unless it starts with a capital letter or `:` (a role such as `:func:`); `"should end with a full stop"`; and `"line N is over 80 characters"` for each such line.

In [ ]:
def news_problems(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_news():\n    \"short, clear, for users\"\n    assert news_problems(\"Fix :func:`textwrap.shorten` dropping a trailing word.\\n\") == []\n    assert news_problems(\":func:`json.dumps` now accepts *sort_keys* everywhere. Patch by Ada Walker.\") == []\n    assert news_problems(\"  \") == [\"empty\"]\n    long = \"fixed a bug \" * 8\n    assert news_problems(f\"{long}\\n\\nAnd another\") == [\"more than one paragraph\", \"should start with a capital letter or a role\",\n                                                       \"should end with a full stop\", \"line 1 is over 80 characters\"]")

### Exercise: Pull request titles

Write `title_problems(title, issue)`: report in this order: `"should start with gh-ISSUE: "` unless it does; `"summary should start with a capital letter"` (the summary is what follows the prefix, or the whole title without one); `"no full stop at the end"` if it ends with one; and `"over 72 characters"` if the whole title is longer.

In [ ]:
def title_problems(title, issue):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_titles():\n    \"gh-N: Summary\"\n    assert title_problems(\"gh-140123: Fix textwrap.shorten dropping a trailing word\", 140123) == []\n    assert title_problems(\"Fix shorten.\", 140123) == [\"should start with gh-140123: \", \"no full stop at the end\"]\n    assert title_problems(\"gh-140124: fix shorten\", 140123) == [\"should start with gh-140123: \", \"summary should start with a capital letter\"]\n    assert title_problems(\"gh-1: \" + \"A\" * 70, 1) == [\"over 72 characters\"]")

### Exercise: Which branches?

Write `backport_labels(change, branches)`: `branches` maps a version such as `"3.13"` to its status (`"feature"`, `"bugfix"`, `"security"` or `"end-of-life"`). A `"feature"` change gets no backports; a `"bug"` or `"docs"` change is backported to every bugfix branch; a `"security"` change to every bugfix and security branch. Return labels `"needs backport to VERSION"`, newest version first (compare versions as numbers). Raise `ValueError` for any other change type.

In [ ]:
def backport_labels(change, branches):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("BRANCHES = {\"3.15\": \"feature\", \"3.14\": \"bugfix\", \"3.13\": \"bugfix\", \"3.12\": \"security\", \"3.11\": \"security\", \"3.10\": \"security\", \"3.9\": \"end-of-life\"}\n\ndef test_labels():\n    \"features stay on main\"\n    assert backport_labels(\"feature\", BRANCHES) == []\n    assert backport_labels(\"bug\", BRANCHES) == [\"needs backport to 3.14\", \"needs backport to 3.13\"]\n    assert backport_labels(\"docs\", BRANCHES) == backport_labels(\"bug\", BRANCHES)\n    assert backport_labels(\"security\", BRANCHES) == [\"needs backport to 3.14\", \"needs backport to 3.13\", \"needs backport to 3.12\",\n                                                     \"needs backport to 3.11\", \"needs backport to 3.10\"]\n    try:\n        backport_labels(\"refactor\", BRANCHES)\n    except ValueError:\n        pass\n    else:\n        raise AssertionError(\"refactor\")")

### Bug hunt: Colons in a file name

`isoformat` puts colons in the time, which aren't allowed in Windows file names and aren't blurb's format: this prints `2026-11-20T14:03:12.gh-issue-140123.AbCdEf.rst`. Format the time with hyphens, so it prints `2026-11-20-14-03-12.gh-issue-140123.AbCdEf.rst`.

In [ ]:
from datetime import datetime

when = datetime(2026, 11, 20, 14, 3, 12)
print(f"{when.isoformat()}.gh-issue-140123.AbCdEf.rst")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"YYYY-MM-DD-HH-MM-SS\"\n    expect_output(\"2026-11-20-14-03-12.gh-issue-140123.AbCdEf.rst\")")

### Bug hunt: 3.9 is newer than 3.13?

Sorting version strings as text puts `3.9` first, so this prints `['3.9', '3.14', '3.13']`. Sort them as numbers, newest first, so it prints `['3.14', '3.13', '3.9']`.

In [ ]:
versions = ["3.13", "3.9", "3.14"]
print(sorted(versions, reverse=True))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"tuples of ints\"\n    expect_output(\"['3.14', '3.13', '3.9']\")")

### Puzzle: Building a NEWS path

Arrange `news_path` (without its checks): the folder for the section, then the date-stamped file name. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the folder of CPython's standard library written in Python (with its slash)
q = ___   # the C file with the built-in list type, as a path
r = ___   # the tool that writes NEWS entries
s = ___   # True or False: new features are backported to bugfix branches
t = ___   # the label a core developer adds when a change needs no NEWS entry

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"Lib/\" and q == \"Objects/listobject.c\" and r == \"blurb\", \"p, q and r\"\n    assert s is False and t == \"skip news\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: ready for review?

Write `pr_checklist(pr, branches)` with today's rules (`title_problems` and `backport_labels` are given). `pr` has `title`, `issue`, `change` (a change type), `files` (paths changed), `labels`, `cla` (a bool) and `ci` (`"passed"`, `"failed"` or `"pending"`). Return `(lines, verdict)`:

1. `"title: ok"` or `"title: "` and its problems joined by `"; "`;
2. `"tests: ok"` if a changed file is under `Lib/test/` or the change is `docs`, else `"tests: no test changed"`;
3. `"news: ok"` if a changed file is under `Misc/NEWS.d/` and contains `gh-issue-ISSUE.`, or the labels include `skip news`; `"news: entry is for another issue"` if there's a NEWS file but not for this issue; else `"news: missing"`;
4. `"docs: ok"`, or `"docs: a new feature needs documentation"` for a feature with no changed file under `Doc/`;
5. `"cla: ok"` or `"cla: not signed"`;
6. `"ci: "` and its status;
7. `"backports: "` and the labels joined by `", "`, or `"backports: none"`.

`verdict` is `"ready for review"` if checks 1 to 5 are ok and CI passed; otherwise `"not ready: "` and the names of the failing checks (`title`, `tests`, `news`, `docs`, `cla`, `ci`) joined by `", "`.

In [ ]:
def title_problems(title, issue):
    prefix = f"gh-{issue}: "
    problems = []
    if title.startswith(prefix):
        summary = title[len(prefix):]
    else:
        problems.append(f"should start with {prefix}")
        summary = title.split(": ", 1)[1] if title.startswith("gh-") and ": " in title else title
    if not summary[:1].isupper():
        problems.append("summary should start with a capital letter")
    if title.endswith("."):
        problems.append("no full stop at the end")
    if len(title) > 72:
        problems.append("over 72 characters")
    return problems

TARGETS = {"feature": set(), "bug": {"bugfix"}, "docs": {"bugfix"}, "security": {"bugfix", "security"}}

def backport_labels(change, branches):
    versions = [v for v, status in branches.items() if status in TARGETS[change]]
    versions.sort(key=lambda v: tuple(int(p) for p in v.split(".")), reverse=True)
    return [f"needs backport to {v}" for v in versions]

def pr_checklist(pr, branches):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("BRANCHES = {\"3.15\": \"feature\", \"3.14\": \"bugfix\", \"3.13\": \"bugfix\", \"3.12\": \"security\"}\n\ndef test_ready():\n    \"a good first pull request\"\n    pr = {\"title\": \"gh-140123: Fix textwrap.shorten dropping a trailing word\", \"issue\": 140123, \"change\": \"bug\",\n          \"files\": [\"Lib/textwrap.py\", \"Lib/test/test_textwrap.py\", \"Misc/NEWS.d/next/Library/2026-11-20-14-03-12.gh-issue-140123.AbCdEf.rst\"],\n          \"labels\": [], \"cla\": True, \"ci\": \"passed\"}\n    lines, verdict = pr_checklist(pr, BRANCHES)\n    assert lines == [\"title: ok\", \"tests: ok\", \"news: ok\", \"docs: ok\", \"cla: ok\", \"ci: passed\",\n                     \"backports: needs backport to 3.14, needs backport to 3.13\"]\n    assert verdict == \"ready for review\"\n\ndef test_not_ready():\n    \"everything a first pull request gets wrong\"\n    pr = {\"title\": \"added sort_keys.\", \"issue\": 140200, \"change\": \"feature\",\n          \"files\": [\"Lib/json/encoder.py\", \"Misc/NEWS.d/next/Library/2026-11-21-09-00-00.gh-issue-140123.QwErTy.rst\"],\n          \"labels\": [], \"cla\": False, \"ci\": \"failed\"}\n    lines, verdict = pr_checklist(pr, BRANCHES)\n    assert lines == [\"title: should start with gh-140200: ; summary should start with a capital letter; no full stop at the end\",\n                     \"tests: no test changed\", \"news: entry is for another issue\", \"docs: a new feature needs documentation\",\n                     \"cla: not signed\", \"ci: failed\", \"backports: none\"]\n    assert verdict == \"not ready: title, tests, news, docs, cla, ci\"\n    docs_pr = {\"title\": \"gh-9: Clarify json.dumps\", \"issue\": 9, \"change\": \"docs\", \"files\": [\"Doc/library/json.rst\"],\n               \"labels\": [\"skip news\"], \"cla\": True, \"ci\": \"pending\"}\n    assert pr_checklist(docs_pr, BRANCHES)[1] == \"not ready: ci\"\n    assert pr_checklist(dict(docs_pr, labels=[]), BRANCHES)[0][2] == \"news: missing\"")

## Warm-up

The standard library module you import as `json` is a folder of Python files, `Lib/json/` in CPython's repository. Much of the standard library is plain Python you can read today, which is why so many first contributions are there, along with tests and documentation.

## The repository

CPython lives at github.com/python/cpython. The parts you'll meet first:

- `Lib/`: the standard library written in Python (`Lib/json/`, `Lib/textwrap.py`).
- `Lib/test/`: the test suite, run by `python -m test`: `Lib/test/test_textwrap.py`, or a package such as `Lib/test/test_json/`.
- `Doc/`: the documentation in reStructuredText, built with Sphinx: `Doc/library/json.rst` is the page you read on docs.python.org.
- `Objects/`: the built-in types in C (`Objects/listobject.c`, `Objects/dictobject.c`).
- `Python/`: the interpreter itself (`Python/ceval.c` runs bytecode; `Python/bltinmodule.c` has the built-in functions).
- `Modules/`: extension modules in C (`Modules/_json.c` speeds up `json`).
- `Misc/NEWS.d/`: the changelog, one small file per change.

Python code in `Lib/` follows PEP 8; C code follows PEP 7.

## From issue to merge

The devguide's workflow, in short:

1. **An issue** on GitHub describes the problem (`gh-12345`). For anything beyond a typo, there should be one before a pull request.
2. **Fork, clone and branch**: work on a branch in your fork, never on `main`.
3. **Change and test**: fix the code, add or update a test in `Lib/test/`, run it with `./python -m test test_textwrap`.
4. **NEWS entry**: describe the change for users with `blurb add` (unless it's trivial; a core developer can add the **skip news** label).
5. **Pull request** against `main`, titled `gh-12345: Fix ...`, with a description that links the issue.
6. **CLA**: a bot checks you've signed the PSF Contributor Licence Agreement (once, online).
7. **Review and CI**: core developers and other contributors review; GitHub Actions runs the tests on several systems. Expect requests for changes, and expect waiting.
8. **Merge** by a core developer, then **backports** to maintained branches, usually by a bot (`miss-islington`).

The details change over time, so always check the current devguide rather than a summary like this one.

## NEWS entries

`blurb add` writes a file under `Misc/NEWS.d/next/SECTION/`, named with the date and time, the issue number and a short random **nonce**, for example `Misc/NEWS.d/next/Library/2026-11-20-14-03-12.gh-issue-140123.AbCdEf.rst`. The sections include `Library`, `Core_and_Builtins`, `Documentation`, `Tests`, `Build`, `Security`, `C_API`, `IDLE`, `Windows`, `macOS` and `Tools-Demos`. The entry is a short reStructuredText paragraph for users, usually starting with what changed: ``Fix :func:`textwrap.shorten` dropping a trailing word.`` Many end with "Patch by" and the author's name.

## Branches and backports

`main` is the next feature release. A few older versions are maintained: the most recent are **bugfix** branches (they get bug fixes and documentation fixes), older ones are **security** branches (security fixes only), and the oldest are **end of life** (nothing). New features go into `main` only. A bug fix is backported to the bugfix branches; a security fix to the security branches too. Backports are requested with labels such as `needs backport to 3.14`. The status of each version is published on the devguide's "Status of Python versions" page.

## Practice

## Quiz

## Daily challenge

## Your turn on CPython

1. **Read the devguide's "Getting started" and "Pull request lifecycle"** pages (devguide.python.org), noting anything that differs from today's summary.
2. **Fork and clone** github.com/python/cpython on your own computer, and find `Lib/textwrap.py`, its tests and its documentation page with today's map.
3. **Read one merged pull request** labelled `type-bug` or `docs`: its issue, its diff, its NEWS entry, its review comments, and its backports.
4. **Sign the PSF CLA** (it takes a few minutes, online), so it's done before your first pull request.

## Before you go

Tomorrow you'll build CPython from source and run its test suite, so you can test a change on the interpreter you changed.